In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: AUSTRALIAN STANDARD CLASSIFICATION OF RELIGIOUS GROUPS
# 
# ARCHITECTURE NOTE: ASCRG is a strict mathematical taxonomy published as a 
# flat spreadsheet. Concepts are organized purely by string length: 
# - 2-digit codes = Broad groups
# - 4-digit codes = Narrow groups
# - 6-digit codes = Religious groups
# 
# Furthermore, ASCRG is not published as Linked Open Data (LOD) and does not 
# assign resolvable URIs to its concepts.
#
# SSSOM ALIGNMENT STRATEGY: 
# This script uses Strategy A (Local Bulk Parsing). It reads the specific 
# "Table 1.3" sheet from the Excel file into memory, scans rows for valid 
# digit codes, and maps them to a dictionary. It then executes a second pass, 
# using string slicing (e.g., code[:4]) to mathematically deduce the parent ID 
# and construct the absolute Hierarchy_Path.
#
# To satisfy SSSOM identifier requirements without hallucinating data, we assign 
# a standard CURIE (e.g., ASCRG:253118) to serve as the primary key, but 
# intentionally leave the formal URI column blank to avoid synthesizing fake 
# web addresses.
# ==============================================================================
# INSTRUCTIONS FOR REPRODUCIBILITY
# 1. Navigate to the ABS ASCRG page: 
#    https://www.abs.gov.au/statistics/classifications/australian-standard-classification-religious-groups/latest-release
# 2. Scroll down to the "Data downloads" section.
# 3. Download the Excel data file (e.g., "2025 ASCRG 1.0.xlsx").
# 4. Place the downloaded .xlsx file in your local `data/external/` directory 
#    and ensure the filename matches the `external_file` variable below.
# ==============================================================================

import os
import sys
import re
import pandas as pd
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
external_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "external"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory. Check PYTHONPATH.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "ASCRG"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="Australian Standard Classification of Religious Groups",
    fallback_uri="" # Intentional blank for non-LOD sources
)
SOURCE_NAME = registry_data['Source_Name']

raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
external_file = os.path.join(external_data_dir, "2025 ASCRG 1.0.xlsx")

if not os.path.exists(external_file):
    print(f"[!] CRITICAL ERROR: The bulk file was not found at {external_file}")
    print("Please download the ASCRG Excel file from the ABS website and place it in the data/external/ directory.")
    sys.exit(1)

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
    else:
        print(f"[*] Found existing ASCRG file. Overwriting for fresh bulk harvest...")
        os.remove(raw_ingest_file)

# --- 4. Pass 1: Extraction & Dictionary Building ---
print("[*] Loading ASCRG Excel file (Table 1.3) into memory...")
try:
    # Restrict loading to Table 1.3, forcing strings to prevent dropping leading zeros
    sheet_df = pd.read_excel(external_file, sheet_name="Table 1.3", header=None, dtype=str)
except Exception as e:
    print(f"[!] CRITICAL ERROR: Could not load sheet 'Table 1.3'. Ensure the sheet name is exact. Exception: {e}")
    sys.exit(1)

code_dict = {}

print("[*] Scanning rows for classification codes...")
for index, row in sheet_df.iterrows():
    # Flatten row and drop NaNs/empty strings
    row_values = [str(x).strip() for x in row.values if pd.notna(x) and str(x).strip() != '']
    
    for i, val in enumerate(row_values):
        # Look for 2, 4, or 6 digit codes
        if re.match(r'^\d{2,6}$', val) and len(val) in [2, 4, 6]:
            if i + 1 < len(row_values):
                code = val
                label = row_values[i+1]
                code_dict[code] = label
                break # Move to next row once code is found

print(f"[*] Successfully extracted {len(code_dict)} unique concepts.")

# --- 5. Pass 2: Hierarchy Construction & Schema Mapping ---
print("[*] Reconstructing mathematical hierarchy and mapping to standard schema...")

all_rows = []
for code, label in code_dict.items():
    
    broad_id = code[:2]
    narrow_id = code[:4]
    
    # Safely build hierarchy path
    path_components = []
    if broad_id in code_dict and broad_id != code:
        path_components.append(code_dict[broad_id])
    if narrow_id in code_dict and narrow_id != code:
        path_components.append(code_dict[narrow_id])
    path_components.append(label)
    
    hierarchy_path = " > ".join(path_components)
    
    # Determine immediate parent
    if len(code) == 6:
        parent_id = narrow_id if narrow_id in code_dict else (broad_id if broad_id in code_dict else "")
    elif len(code) == 4:
        parent_id = broad_id if broad_id in code_dict else ""
    else:
        parent_id = "" # Broad groups have no parent

    extracted_data = {
        'Source_System': SOURCE_NAME,
        'Primary_Label': label,
        'CURIE': f"{SOURCE_PREFIX}:{code}",
        'Formal_Label': "", # not provided in ASCRG
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': hierarchy_path,
        'Synonyms': "", # not provided in ASCRG
        'Description': "", # not provided in ASCRG
        'Parent_IDs': parent_id,
        'Concept_ID': code,
        'URI': "", # Intentional Blank: ASCRG is not LOD and has no resolvable URIs
        'Has_Translation': "", # not provided in ASCRG
        'Status': "active",
        'Crosswalks': "" # not provided in ASCRG
    }
    
    all_rows.append(finalize_row(extracted_data))

# --- 6. Export to Bronze Layer ---
if all_rows:
    df = pd.DataFrame(all_rows)[COLUMN_ORDER]
    df.to_csv(raw_ingest_file, index=False, encoding='utf-8-sig')
    print(f"\n[COMPLETE] Successfully wrote {len(all_rows)} concepts to {raw_ingest_file}")
else:
    print("\n[!] CRITICAL ERROR: No data was parsed from the Excel file.")